# Soft-wall quenched log-polar effective Hamiltonian

This local CPU notebook analyzes all 300 completed soft-v3 trajectories in the full $40N_y$ transfer sector. It constructs $A_\xi=\operatorname{arctanh}G_\xi$ before the $y$ twirl and disorder average, compares quenched and typical generators, and tests wall chirality separately from Cardy--Calabrese criticality.

In [ ]:
# Runtime and complete editable analysis configuration. Run this cell first.
import os
from pathlib import Path

CPU_RANGE = ''                 # Example: '0-15' or '0-7,16-23'.
N_WORKERS = 1                 # Parallel trajectories within a five-sample shard.
NY_VALUES = (20, 30, 40)
MAX_SAMPLES_PER_NY = None     # Set to 1 for smoke; None uses all 100 per size.
VERIFY_SOURCE_HASHES = True
POLAR_LOG_CAPS = (8.0, 10.0, 12.0)
DEFAULT_POLAR_LOG_CAP = 10.0
RENYI_ORDERS = (1, 2, 3)
BOUNDARY_TWISTS = (-1.0e-7, 1.0e-7)
BOOTSTRAP_REPLICATES = 2000
JACKKNIFE_GROUPS = 10
OUTPUT_REVISION = 'soft_quenched_log_polar_v1'
OUTPUT_ROOT_OVERRIDE = ''     # Absolute or repo-relative path; blank uses this bundle.

def _parse_cpu_range(spec):
    selected = set()
    for token in spec.replace(' ', '').split(','):
        if not token:
            continue
        if '-' in token:
            lo, hi = (int(value) for value in token.split('-', 1))
            selected.update(range(lo, hi + 1))
        else:
            selected.add(int(token))
    return selected

available_cpus = set(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else set()
if CPU_RANGE and available_cpus:
    requested_cpus = _parse_cpu_range(CPU_RANGE)
    if not requested_cpus or not requested_cpus.issubset(available_cpus):
        raise ValueError(f'CPU_RANGE={CPU_RANGE!r} is not contained in {sorted(available_cpus)}')
    os.sched_setaffinity(0, requested_cpus)
allocated_cpus = sorted(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else []
blas_threads = 1 if N_WORKERS > 1 else max(1, len(allocated_cpus))
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = str(blas_threads)
print({'allocated_cpus': allocated_cpus, 'workers': N_WORKERS, 'blas_threads': blas_threads})

## Theory and estimator order

For every disorder trajectory, the left polar convention is $T_\xi=P_{L,\xi}U_\xi$, $A_\xi=\log P_{L,\xi}=\operatorname{arctanh}G_\xi$, and $h_\xi=-2A_\xi$. The nonlinear logarithm is taken trajectory by trajectory, then each result is twirled along $y$, and only then is the quenched mean formed. Soft walls retain all $x=0,\ldots,19$ columns; wall weights use $(4,5,6)$ and $(14,15,16)$ without truncation. Cardy--Calabrese scaling tests central charge, while localized counterpropagating branches test chirality.

In [ ]:
import json
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

candidate = Path.cwd()
if not (candidate / 'polar_effective_hamiltonian.py').is_file():
    candidate = Path('00_WORKSPACE/CURRENT/final_production_new_designs/07_maxmix_hard_soft_purification').resolve()
BUNDLE_ROOT = candidate
if not (BUNDLE_ROOT / 'polar_effective_hamiltonian.py').is_file():
    raise FileNotFoundError('Could not locate campaign 07')
sys.path.insert(0, str(BUNDLE_ROOT))
import polar_effective_hamiltonian as polar

SPEC = polar.SOFT_SPEC
DATA_ROOT = SPEC.data_root
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve() if OUTPUT_ROOT_OVERRIDE else BUNDLE_ROOT / 'analysis_outputs' / OUTPUT_REVISION
CACHE_ROOT = OUTPUT_ROOT / 'sample_cache'
FIGURE_ROOT = OUTPUT_ROOT / 'figure_assets'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = polar.AnalysisConfig(construction='soft', caps=POLAR_LOG_CAPS, default_cap=DEFAULT_POLAR_LOG_CAP, renyi_orders=RENYI_ORDERS, twists=BOUNDARY_TWISTS, bootstrap_replicates=BOOTSTRAP_REPLICATES, jackknife_groups=JACKKNIFE_GROUPS)
mpl.rcParams.update({'font.family': 'CMU Sans Serif', 'font.size': 8, 'axes.linewidth': 0.8})
print(json.dumps({'data_root': str(DATA_ROOT), 'output_root': str(OUTPUT_ROOT), 'analysis_config': CONFIG.identity()}, indent=2))

## Exact coupled-domain-wall calibration

Before touching campaign data, reproduce the accepted untruncated exact-B0 estimator calibration at $N_x=20,N_y=48$.

In [ ]:
B0_TABLE = BUNDLE_ROOT.parents[1] / 'experiment_review' / 'b0_exact_domain_wall' / 'results' / '20260816_183928' / 'processed' / 'tables' / 'entropy_fits.csv'
b0 = pd.read_csv(B0_TABLE)
b0_check = b0[(b0.construction == 'coupled') & (b0.geometry_key == 'coupled__Nx020__Ny048') & (b0.quantity == 'full_strip')].copy()
if set(b0_check.q) != {1, 2, 3} or np.max(np.abs(b0_check.c_estimate - 1.0)) >= 0.02:
    raise RuntimeError('Exact coupled B0 calibration did not reproduce c_w=1')
display(b0_check[['q', 'slope', 'c_estimate', 'r2', 'ay_min', 'ay_max']])

## Verify the completed soft-v3 inventory

All result/completion pairs are checked before analysis. The displayed contract is read from the identity that was verified against every shard.

In [ ]:
records = polar.discover_samples(DATA_ROOT, construction='soft', ny_values=NY_VALUES, verify_hashes=VERIFY_SOURCE_HASHES, require_complete=True)
inventory = pd.DataFrame([{'Ny': row.ny, 'sample': row.sample_index, 'source': row.result_path.name} for row in records])
display(inventory.groupby('Ny').agg(samples=('sample', 'count'), first=('sample', 'min'), last=('sample', 'max')))
campaign_parameters = {'sampling_revision': SPEC.revision, 'configuration_hash': SPEC.configuration_hash, 'construction': SPEC.construction, 'Nx': polar.EXPECTED_NX, 'Ny': list(NY_VALUES), 'cycles': {str(ny): 4 * ny for ny in NY_VALUES}, 'dtype': 'complex128', 'active_x': list(SPEC.active_x), 'active_dimension': {str(ny): SPEC.transfer_mode_count(ny) for ny in NY_VALUES}, 'trajectory_count': len(records)}
print(json.dumps(campaign_parameters, indent=2))

## Build or resume trajectory caches

Each checksum-bound cache is committed with a completion JSON. Restarting skips valid trajectories and repeats only an interrupted trajectory.

In [ ]:
cache_report = polar.build_sample_caches(records, cache_root=CACHE_ROOT, config=CONFIG, progress=tqdm, max_samples_per_ny=MAX_SAMPLES_PER_NY, workers=N_WORKERS)
print('[cache completion]', json.dumps(cache_report, indent=2))

## Aggregate quenched, typical, annealed, and flattened analyses

In [ ]:
analysis = polar.aggregate_analysis(records, cache_root=CACHE_ROOT, output_root=OUTPUT_ROOT, config=CONFIG, max_samples_per_ny=MAX_SAMPLES_PER_NY)
summary = analysis['summary']
spectrum_df = pd.read_csv(OUTPUT_ROOT / 'spectrum_rows.csv')
velocity_df = pd.read_csv(OUTPUT_ROOT / 'velocity_summary.csv')
entropy_df = pd.read_csv(OUTPUT_ROOT / 'entropy_curves.csv')
fit_df = pd.read_csv(OUTPUT_ROOT / 'entropy_fit_summary.csv')
diagnostics_df = pd.read_csv(OUTPUT_ROOT / 'numerical_diagnostics.csv')
arrays = np.load(OUTPUT_ROOT / 'notebook_arrays.npz', allow_pickle=False)
print(json.dumps(summary['claim_ledger'], indent=2))

## Quenched and typical wall spectra

The same overlap-tracked branches are shown as the endpoint generator and as the per-cycle rate.

In [ ]:
colors = ('#d62728', '#1f77b4')
markers = ('^', 'o')
fig, axes = plt.subplots(2, len(NY_VALUES), figsize=(7.05, 4.35), sharex='col')
for panel, ny in enumerate(NY_VALUES):
    key = f'Ny{ny:03d}_cap{DEFAULT_POLAR_LOG_CAP:g}'
    ky = arrays[f'{key}_quenched_ky']; low = arrays[f'{key}_quenched_low_energy']
    qwall = arrays[f'{key}_quenched_wall_energy']; twall = arrays[f'{key}_typical_wall_energy_mean']
    for row, scale in enumerate((1.0, 1.0 / (4 * ny))):
        ax = axes[row, panel]
        for band in range(low.shape[1]): ax.plot(ky, scale * low[:, band], '.', ms=2, color='0.72')
        for wall in range(2):
            ax.plot(ky, scale * qwall[:, wall], color=colors[wall], marker=markers[wall], ms=2.7, lw=1, label=f'wall {wall+1}, quenched')
            ax.plot(ky, scale * twall[:, wall], color=colors[wall], ls='--', lw=0.9, label=f'wall {wall+1}, typical')
        ax.axhline(0, color='black', ls=':', lw=0.8); ax.tick_params(direction='in', top=True, right=True)
    axes[0, panel].set_title(rf'$N_y={ny}$'); axes[1, panel].set_xlabel(r'$k_y$')
axes[0, 0].set_ylabel(r'$h_{\mathrm{eff}}=-2\log P_L$'); axes[1, 0].set_ylabel(r'$h_{\mathrm{eff}}/(4N_y)$')
axes[0, -1].legend(frameon=False, fontsize=5.5); fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'soft_quenched_typical_wall_spectrum.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'soft_quenched_typical_wall_spectrum.png', dpi=300, bbox_inches='tight')
plt.show()

## Localized branch dispersions and velocities

Opposite velocity signs at branches carrying at least 50% weight in the full three-column wall windows are the chirality criterion.

In [ ]:
fig, axes = plt.subplots(1, len(NY_VALUES), figsize=(7.05, 2.35), sharey=True)
for panel, (ax, ny) in enumerate(zip(np.atleast_1d(axes), NY_VALUES)):
    for ci, cap in enumerate(POLAR_LOG_CAPS):
        velocity = arrays[f'Ny{ny:03d}_cap{cap:g}_quenched_velocity']
        for wall in range(2): ax.plot(cap + (wall - .5) * .10, velocity[wall], marker=markers[wall], color=colors[wall], ms=5, ls='none', label=f'wall {wall+1}' if ci == 0 else None)
    ax.axhline(0, color='black', ls='--', lw=.8); ax.set_title(rf'$N_y={ny}$'); ax.set_xlabel(r'polar cap $\Lambda$'); ax.tick_params(direction='in', top=True, right=True)
axes[0].set_ylabel(r'$\partial h_{\mathrm{eff}}/\partial k_y$'); axes[-1].legend(frameon=False, fontsize=7); fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'soft_wall_velocity_cap_sensitivity.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'soft_wall_velocity_cap_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

## Cardy--Calabrese entropy curves

Whole-trajectory intervals accompany endpoint and typical curves; quenched-mean uncertainty is supplied by ten-group delete-group jackknife.

In [ ]:
family_style = {'actual_endpoint': ('#1f77b4', 'o', '-', 'actual endpoint'), 'typical_sample': ('#2ca02c', 's', '--', 'typical ground state'), 'quenched_mean': ('#d62728', '^', ':', 'quenched ground state')}
fig, axes = plt.subplots(1, len(NY_VALUES), figsize=(7.05, 2.55))
for ax, ny in zip(np.atleast_1d(axes), NY_VALUES):
    for family, (color, marker, linestyle, label) in family_style.items():
        sub = entropy_df[(entropy_df.family == family) & (entropy_df.Ny == ny) & (entropy_df.renyi_order == 1)]
        if family != 'actual_endpoint': sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
        sub = sub.sort_values('Ay'); x = polar.log_chord(ny, sub.Ay.to_numpy())
        ax.plot(x, sub['mean'], color=color, marker=marker, ms=3, lw=1, ls=linestyle, label=label)
        ax.fill_between(x, sub.ci_low, sub.ci_high, color=color, alpha=.10, lw=0)
    ax.set_title(rf'$N_y={ny}$'); ax.set_xlabel(r'$\log[(N_y/\pi)\sin(\pi A_y/N_y)]$'); ax.tick_params(direction='in', top=True, right=True)
axes[0].set_ylabel(r'$S_1(A_y)$'); axes[-1].legend(frameon=False, fontsize=6); fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'soft_cardy_calabrese_entropy_curves.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'soft_cardy_calabrese_entropy_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## Extracted central charge by Renyi order

In [ ]:
fig, axes = plt.subplots(1, len(RENYI_ORDERS), figsize=(7.05, 2.45), sharey=True)
for ax, order in zip(np.atleast_1d(axes), RENYI_ORDERS):
    for family, (color, marker, linestyle, label) in family_style.items():
        sub = fit_df[(fit_df.family == family) & (fit_df.renyi_order == order) & (fit_df.fit_variant == 'primary')]
        if family != 'actual_endpoint': sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
        sub = sub.sort_values('Ny'); central = sub.c_per_wall.to_numpy(dtype=float)
        low = pd.to_numeric(sub.ensemble_c_ci_low).to_numpy(dtype=float); high = pd.to_numeric(sub.ensemble_c_ci_high).to_numpy(dtype=float)
        yerr = np.vstack((np.maximum(0, central - low), np.maximum(0, high - central)))
        ax.errorbar(sub.Ny, central, yerr=yerr, color=color, marker=marker, ms=4, lw=1, ls=linestyle, capsize=2, label=label)
    ax.axhline(1, color='black', ls='--', lw=.8); ax.set_title(rf'$n={order}$'); ax.set_xlabel(r'$N_y$'); ax.set_xticks(NY_VALUES); ax.tick_params(direction='in', top=True, right=True)
axes[0].set_ylabel(r'$c_{w,n}$'); axes[-1].legend(frameon=False, fontsize=6); fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'soft_renyi_central_charge_comparison.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'soft_renyi_central_charge_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Cap, estimator-order, fit-window, and twist controls

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.05, 5.0)); ny = max(NY_VALUES)
ax = axes[0, 0]
for cap in POLAR_LOG_CAPS:
    sub = fit_df[(fit_df.family == 'quenched_mean') & (fit_df.Ny == ny) & (fit_df.renyi_order == 1) & (fit_df.fit_variant == 'primary')]
    sub = sub[np.isclose(pd.to_numeric(sub.cap), cap) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
    if len(sub): ax.plot(cap, sub.iloc[0].c_per_wall, 'o', color='#d62728')
ax.set_title('polar-cap sensitivity'); ax.set_xlabel(r'$\Lambda$'); ax.set_ylabel(r'$c_{w,1}$')
ax = axes[0, 1]
for family, label in (('quenched_mean', 'quenched'), ('annealed_covariance', 'annealed'), ('flattened_parent', 'flattened')):
    sub = fit_df[(fit_df.family == family) & (fit_df.Ny == ny) & (fit_df.renyi_order == 1) & (fit_df.fit_variant == 'primary')]
    if family == 'quenched_mean': sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
    if len(sub): ax.plot([label], [sub.iloc[0].c_per_wall], 'o')
ax.set_title('estimator-order controls'); ax.tick_params(axis='x', rotation=15)
ax = axes[1, 0]; variants = ('primary', 'aymin6', 'drop_endpoint')
for family, (color, marker, _, label) in family_style.items():
    sub = fit_df[(fit_df.family == family) & (fit_df.Ny == ny) & (fit_df.renyi_order == 1)]
    if family != 'actual_endpoint': sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
    ax.plot(variants, [sub[sub.fit_variant == variant].iloc[0].c_per_wall for variant in variants], marker=marker, color=color, label=label)
ax.set_title('fit-window sensitivity'); ax.tick_params(axis='x', rotation=15)
ax = axes[1, 1]
for family, color, marker in (('quenched_mean', '#d62728', '^'), ('typical_sample', '#2ca02c', 's')):
    sub = fit_df[(fit_df.family == family) & (fit_df.Ny == ny) & (fit_df.renyi_order == 1) & (fit_df.fit_variant == 'primary')]
    sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP)].sort_values('twist')
    ax.plot(sub.twist * 1e7, sub.c_per_wall, marker=marker, color=color, label=family)
ax.set_title('twist sensitivity'); ax.set_xlabel(r'$\phi\times10^7$'); ax.legend(frameon=False, fontsize=6)
for ax in axes.flat: ax.axhline(1, color='black', ls='--', lw=.8); ax.tick_params(direction='in', top=True, right=True)
fig.tight_layout(); fig.savefig(FIGURE_ROOT / 'soft_cap_estimator_fit_twist_controls.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'soft_cap_estimator_fit_twist_controls.png', dpi=300, bbox_inches='tight'); plt.show()

## Raw diagnostics and claim ledger

This final cell exposes fit rows, numerical maxima, saturation counts, and all gated interpretations.

In [ ]:
display(fit_df.sort_values(['family', 'Ny', 'renyi_order', 'fit_variant']).reset_index(drop=True))
numeric_columns = [column for column in diagnostics_df.columns if column.endswith('residual') or column.endswith('coupling')]
display(diagnostics_df[numeric_columns].max(numeric_only=True).to_frame('maximum'))
display(diagnostics_df.groupby(['family', 'Ny', 'cap'], dropna=False).agg(rows=('Ny', 'size'), interior_modes=('interior_mode_count', 'median'), capped_modes=('capped_mode_count', 'median')).reset_index())
display(velocity_df)
print(json.dumps(summary['claim_ledger'], indent=2))
print(f'analysis complete: {OUTPUT_ROOT}')